<a href="https://colab.research.google.com/github/akaleniuszka/03MIAR--Algoritmo-de-Optimizacion/blob/main/AG3/AG3_Actividad_Guiada_TSP_Alfredo_Kaleniuszka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AG3 - Actividad Guiada 3

**Asignatura:** Algoritmos de Optimización

**Alumno/a:** Alfredo Kaleniuszka

**URL GitHub:**

## Objetivo de la actividad

Según las diapositivas de la AG3, se trabaja sobre el problema del viajante de comercio
(`swiss42.tsp`) y se desarrollan las siguientes técnicas:

- Búsqueda aleatoria.
- Búsqueda local.
- Recocido simulado.
- Colonia de hormigas, indicada como no evaluable.

La instancia `swiss42.tsp` tiene 42 ciudades y la mejor solución conocida indicada en el
material es de distancia **1273**. Esa referencia se usa al final para comparar los resultados.


## 1. Carga de librerías y datos

El PDF propone descargar `swiss42.tsp` desde TSPLIB. Como la nota de la actividad indica que,
si la descarga falla, debe usarse el fichero `.tsp` adjunto, la celda siguiente intenta primero
descargarlo y, si no puede, usa el archivo local `swiss42.tsp`.

Para evitar problemas de instalación en distintos entornos, se lee directamente la matriz de
distancias del fichero TSPLIB.


In [1]:
import gzip
import math
import random
import urllib.request
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
random.seed(42)

FICHERO = Path("/content/drive/MyDrive/Master IA - VIU/Algoritmos de Optimización/swiss42.tsp") # Cambiar si fuera necesario

if not FICHERO.exists():
    try:
        print("Descargando swiss42.tsp desde TSPLIB...")
        gz_path = Path("swiss42.tsp.gz")
        urllib.request.urlretrieve(URL, gz_path)
        with gzip.open(gz_path, "rb") as origen:
            FICHERO.write_bytes(origen.read())
        print("Descarga completada.")
    except Exception as e:
        raise FileNotFoundError(
            "No se pudo descargar el fichero. Sube el archivo swiss42.tsp adjunto a la actividad."
        ) from e

def cargar_tsp_matriz(ruta):
    lineas = Path(ruta).read_text(encoding="utf-8").splitlines()
    dimension = None
    inicio = None

    for i, linea in enumerate(lineas):
        texto = linea.strip()
        if texto.startswith("DIMENSION"):
            dimension = int(texto.replace(":", " ").split()[-1])
        elif texto.startswith("EDGE_WEIGHT_SECTION"):
            inicio = i + 1
            break

    if dimension is None or inicio is None:
        raise ValueError("El fichero no tiene el formato esperado de TSPLIB.")

    valores = []
    for linea in lineas[inicio:]:
        texto = linea.strip()
        if texto and texto != "EOF":
            valores.extend(int(x) for x in texto.split())

    if len(valores) != dimension * dimension:
        raise ValueError("La matriz de distancias no tiene el tamaño esperado.")

    return [
        valores[i * dimension:(i + 1) * dimension]
        for i in range(dimension)
    ]

matriz = cargar_tsp_matriz(FICHERO)
Nodos = list(range(len(matriz)))
OPTIMO_CONOCIDO = 1273

print(f"Instancia cargada: {FICHERO.name}")
print(f"Número de ciudades: {len(Nodos)}")
print(f"Mejor solución conocida: {OPTIMO_CONOCIDO}")


Mounted at /content/drive
Instancia cargada: swiss42.tsp
Número de ciudades: 42
Mejor solución conocida: 1273


## 2. Funciones básicas

Se representa una solución como una permutación de los nodos. Para reducir soluciones equivalentes
por rotación, todas las rutas comienzan en el nodo `0`, tal como se indica en las diapositivas.


In [2]:
def crear_solucion(Nodos):
    solucion = [Nodos[0]]
    pendientes = Nodos[1:].copy()
    random.shuffle(pendientes)
    return solucion + pendientes

def distancia(a, b, matriz):
    return matriz[a][b]

def distancia_total(solucion, matriz):
    total = 0
    for i in range(len(solucion) - 1):
        total += distancia(solucion[i], solucion[i + 1], matriz)
    total += distancia(solucion[-1], solucion[0], matriz)
    return total

def imprimir_resultado(nombre, solucion, coste, extra=None):
    gap = ((coste - OPTIMO_CONOCIDO) / OPTIMO_CONOCIDO) * 100
    print(nombre)
    print("-" * len(nombre))
    print(f"Distancia total: {coste}")
    print(f"Gap respecto a 1273: {gap:.2f}%")
    if extra is not None:
        print(extra)
    print(f"Ruta: {solucion}")
    print()

sol_prueba = crear_solucion(Nodos)
imprimir_resultado("Solución aleatoria de prueba", sol_prueba, distancia_total(sol_prueba, matriz))


Solución aleatoria de prueba
----------------------------
Distancia total: 4543
Gap respecto a 1273: 256.87%
Ruta: [0, 10, 4, 11, 37, 26, 25, 31, 12, 5, 33, 22, 29, 24, 19, 13, 35, 36, 27, 14, 23, 21, 32, 38, 30, 20, 17, 40, 34, 3, 1, 39, 28, 6, 7, 9, 15, 16, 18, 2, 8, 41]



## 3. Búsqueda aleatoria

La búsqueda aleatoria genera soluciones al azar durante un número fijo de iteraciones y conserva
la mejor. Es una técnica sencilla que sirve como referencia inicial.


In [3]:
def busqueda_aleatoria(matriz, iteraciones):
    mejor_solucion = None
    mejor_distancia = float("inf")

    for _ in range(iteraciones):
        solucion = crear_solucion(Nodos)
        coste = distancia_total(solucion, matriz)

        if coste < mejor_distancia:
            mejor_solucion = solucion
            mejor_distancia = coste

    return mejor_solucion, mejor_distancia

sol_aleatoria, dist_aleatoria = busqueda_aleatoria(matriz, 10000)
imprimir_resultado("Búsqueda aleatoria", sol_aleatoria, dist_aleatoria)


Búsqueda aleatoria
------------------
Distancia total: 3624
Gap respecto a 1273: 184.68%
Ruta: [0, 29, 28, 32, 20, 2, 10, 27, 40, 38, 39, 7, 17, 11, 30, 4, 3, 16, 19, 6, 18, 9, 21, 23, 22, 8, 24, 12, 26, 13, 37, 14, 36, 25, 41, 34, 33, 1, 35, 31, 5, 15]



## 4. Búsqueda local

La búsqueda local intensifica la exploración alrededor de una solución inicial. En este caso se
usa una vecindad 2-opt: se eligen dos posiciones y se invierte el segmento intermedio. Con 42
nodos y el nodo 0 fijo, se revisan `41*40/2 = 820` vecinos posibles.


In [4]:
def genera_vecina_2opt(solucion, matriz):
    mejor_vecina = solucion[:]
    mejor_distancia = distancia_total(solucion, matriz)

    for i in range(1, len(solucion) - 1):
        for j in range(i + 1, len(solucion)):
            vecina = solucion[:i] + solucion[i:j + 1][::-1] + solucion[j + 1:]
            coste = distancia_total(vecina, matriz)

            if coste < mejor_distancia:
                mejor_vecina = vecina
                mejor_distancia = coste

    return mejor_vecina, mejor_distancia

def busqueda_local(matriz, solucion_inicial=None, max_iter=1000):
    solucion = solucion_inicial[:] if solucion_inicial else crear_solucion(Nodos)
    mejor_distancia = distancia_total(solucion, matriz)

    for iteracion in range(1, max_iter + 1):
        vecina, distancia_vecina = genera_vecina_2opt(solucion, matriz)

        if distancia_vecina >= mejor_distancia:
            return solucion, mejor_distancia, iteracion

        solucion = vecina
        mejor_distancia = distancia_vecina

    return solucion, mejor_distancia, max_iter

sol_local, dist_local, it_local = busqueda_local(matriz, sol_aleatoria)
imprimir_resultado(
    "Búsqueda local",
    sol_local,
    dist_local,
    extra=f"Iteraciones hasta mínimo local: {it_local}"
)


Búsqueda local
--------------
Distancia total: 1373
Gap respecto a 1273: 7.86%
Iteraciones hasta mínimo local: 34
Ruta: [0, 3, 4, 2, 27, 28, 32, 34, 33, 20, 31, 35, 36, 17, 7, 37, 15, 16, 14, 19, 13, 12, 11, 25, 41, 23, 9, 21, 40, 24, 39, 22, 38, 30, 29, 8, 10, 18, 26, 5, 6, 1]



## 5. Recocido simulado

El recocido simulado permite aceptar soluciones peores con una probabilidad dependiente de la
temperatura y de la diferencia de costes. Al principio, con temperatura alta, se aceptan más
movimientos peores; al final, con temperatura baja, el algoritmo se vuelve más restrictivo.


In [5]:
def genera_vecina_aleatoria(solucion):
    i, j = sorted(random.sample(range(1, len(solucion)), 2))
    return solucion[:i] + solucion[i:j + 1][::-1] + solucion[j + 1:]

def aceptar_peor(delta, temperatura):
    return random.random() < math.exp(-delta / temperatura)

def bajar_temperatura(temperatura, factor=0.995):
    return temperatura * factor

def recocido_simulado(
    matriz,
    temperatura_inicial=10000,
    temperatura_final=0.001,
    factor_enfriamiento=0.995,
    iteraciones_por_temperatura=25
):
    solucion_actual = crear_solucion(Nodos)
    distancia_actual = distancia_total(solucion_actual, matriz)

    mejor_solucion = solucion_actual[:]
    mejor_distancia = distancia_actual

    temperatura = temperatura_inicial
    iteraciones = 0

    while temperatura > temperatura_final:
        for _ in range(iteraciones_por_temperatura):
            vecina = genera_vecina_aleatoria(solucion_actual)
            distancia_vecina = distancia_total(vecina, matriz)
            delta = distancia_vecina - distancia_actual

            if delta < 0 or aceptar_peor(delta, temperatura):
                solucion_actual = vecina
                distancia_actual = distancia_vecina

                if distancia_actual < mejor_distancia:
                    mejor_solucion = solucion_actual[:]
                    mejor_distancia = distancia_actual

            iteraciones += 1

        temperatura = bajar_temperatura(temperatura, factor_enfriamiento)

    return mejor_solucion, mejor_distancia, iteraciones

sol_recocido, dist_recocido, it_recocido = recocido_simulado(matriz)
imprimir_resultado(
    "Recocido simulado",
    sol_recocido,
    dist_recocido,
    extra=f"Iteraciones realizadas: {it_recocido}"
)


Recocido simulado
-----------------
Distancia total: 1315
Gap respecto a 1273: 3.30%
Iteraciones realizadas: 80400
Ruta: [0, 27, 2, 3, 4, 6, 5, 19, 13, 26, 18, 12, 11, 25, 10, 8, 41, 23, 9, 21, 40, 24, 39, 22, 38, 30, 29, 28, 32, 34, 33, 20, 35, 36, 31, 17, 37, 15, 16, 14, 7, 1]



## 6. Colonia de hormigas

Aunque las diapositivas indican que la colonia de hormigas no es evaluable, se incluye una
implementación funcional. La probabilidad de elegir el siguiente nodo combina:

- Feromona acumulada en la arista.
- Inversa de la distancia.
- Parámetros `alpha` y `beta`, que ponderan feromona y visibilidad.


In [6]:
def seleccionar_siguiente(actual, pendientes, feromonas, matriz, alpha, beta):
    pesos = []

    for nodo in pendientes:
        feromona = feromonas[actual][nodo] ** alpha
        visibilidad = (1 / matriz[actual][nodo]) ** beta
        pesos.append(feromona * visibilidad)

    total = sum(pesos)
    if total == 0:
        return random.choice(list(pendientes))

    umbral = random.random() * total
    acumulado = 0
    for nodo, peso in zip(pendientes, pesos):
        acumulado += peso
        if acumulado >= umbral:
            return nodo

    return list(pendientes)[-1]

def construir_ruta_hormiga(matriz, feromonas, alpha, beta):
    ruta = [0]
    pendientes = set(Nodos[1:])

    while pendientes:
        siguiente = seleccionar_siguiente(ruta[-1], pendientes, feromonas, matriz, alpha, beta)
        ruta.append(siguiente)
        pendientes.remove(siguiente)

    return ruta

def colonia_hormigas(
    matriz,
    num_hormigas=42,
    iteraciones=150,
    alpha=1,
    beta=3,
    evaporacion=0.3,
    q=100
):
    n = len(matriz)
    feromonas = [[1.0 for _ in range(n)] for _ in range(n)]
    mejor_solucion = None
    mejor_distancia = float("inf")

    for _ in range(iteraciones):
        rutas = []

        for _ in range(num_hormigas):
            ruta = construir_ruta_hormiga(matriz, feromonas, alpha, beta)
            coste = distancia_total(ruta, matriz)
            rutas.append((ruta, coste))

            if coste < mejor_distancia:
                mejor_solucion = ruta
                mejor_distancia = coste

        for i in range(n):
            for j in range(n):
                feromonas[i][j] *= (1 - evaporacion)

        for ruta, coste in rutas:
            deposito = q / coste
            for i in range(n):
                a = ruta[i]
                b = ruta[(i + 1) % n]
                feromonas[a][b] += deposito
                feromonas[b][a] += deposito

    return mejor_solucion, mejor_distancia

sol_hormigas, dist_hormigas = colonia_hormigas(matriz)
imprimir_resultado("Colonia de hormigas", sol_hormigas, dist_hormigas)


Colonia de hormigas
-------------------
Distancia total: 1338
Gap respecto a 1273: 5.11%
Ruta: [0, 1, 6, 4, 3, 27, 2, 28, 29, 30, 32, 34, 20, 33, 38, 22, 39, 21, 40, 24, 41, 23, 9, 8, 10, 25, 11, 12, 18, 26, 5, 13, 19, 14, 16, 15, 37, 36, 35, 31, 17, 7]



## 7. Mejora opcional: entornos variables

El PDF propone como tarea opcional probar otros generadores de vecindad. A continuación se añade
una búsqueda local con entornos variables: intercambio, inversión de sublista y reinicio parcial.


In [7]:
def vecino_inversion(solucion):
    i, j = sorted(random.sample(range(1, len(solucion)), 2))
    return solucion[:i] + solucion[i:j + 1][::-1] + solucion[j + 1:]

def vecino_intercambio_aleatorio(solucion):
    vecina = solucion[:]
    i, j = random.sample(range(1, len(solucion)), 2)
    vecina[i], vecina[j] = vecina[j], vecina[i]
    return vecina

def vecino_barajado_sublista(solucion):
    i, j = sorted(random.sample(range(1, len(solucion)), 2))
    sublista = solucion[i:j + 1]
    random.shuffle(sublista)
    return solucion[:i] + sublista + solucion[j + 1:]

def busqueda_entornos_variables(matriz, rondas=5000):
    operadores = [vecino_intercambio_aleatorio, vecino_inversion, vecino_barajado_sublista]
    solucion = crear_solucion(Nodos)
    mejor_solucion = solucion[:]
    mejor_distancia = distancia_total(solucion, matriz)

    for k in range(rondas):
        operador = operadores[k % len(operadores)]
        vecina = operador(solucion)
        coste = distancia_total(vecina, matriz)

        if coste < mejor_distancia:
            solucion = vecina
            mejor_solucion = vecina[:]
            mejor_distancia = coste
        elif k % 200 == 0:
            solucion = mejor_solucion[:]

    return mejor_solucion, mejor_distancia

sol_vns, dist_vns = busqueda_entornos_variables(matriz)
imprimir_resultado("Mejora opcional - Entornos variables", sol_vns, dist_vns)


Mejora opcional - Entornos variables
------------------------------------
Distancia total: 1410
Gap respecto a 1273: 10.76%
Ruta: [0, 32, 30, 29, 28, 38, 22, 39, 24, 40, 21, 9, 23, 41, 8, 10, 25, 11, 12, 18, 26, 4, 2, 27, 3, 1, 6, 5, 13, 19, 14, 16, 15, 37, 7, 17, 31, 36, 35, 33, 34, 20]



## 8. Comparación final

La tabla final permite comparar las técnicas desarrolladas frente al valor de referencia de la
instancia `swiss42.tsp`.


In [8]:
resultados = {
    "Búsqueda aleatoria": dist_aleatoria,
    "Búsqueda local": dist_local,
    "Recocido simulado": dist_recocido,
    "Colonia de hormigas": dist_hormigas,
    "Entornos variables opcional": dist_vns,
}

print(f"{'Método':30s} {'Distancia':>10s} {'Gap vs 1273':>12s}")
print("-" * 56)
for metodo, coste in sorted(resultados.items(), key=lambda x: x[1]):
    gap = ((coste - OPTIMO_CONOCIDO) / OPTIMO_CONOCIDO) * 100
    print(f"{metodo:30s} {coste:10.0f} {gap:11.2f}%")


Método                          Distancia  Gap vs 1273
--------------------------------------------------------
Recocido simulado                    1315        3.30%
Colonia de hormigas                  1338        5.11%
Búsqueda local                       1373        7.86%
Entornos variables opcional          1410       10.76%
Búsqueda aleatoria                   3624      184.68%


## Conclusiones

La búsqueda aleatoria proporciona una solución inicial, pero no explota la estructura del problema.
La búsqueda local mejora de forma importante porque intensifica alrededor de una solución mediante
intercambios. El recocido simulado puede escapar de mínimos locales al aceptar movimientos peores
durante las primeras fases. La colonia de hormigas construye rutas usando memoria colectiva mediante
feromonas y cercanía entre ciudades.

En general, las técnicas que combinan intensificación y diversificación se acercan más al valor de
referencia de `1273` que la búsqueda aleatoria pura.
